In [1]:
# Set WD at root
import sys
import os

root_path = os.path.abspath(os.path.join('..'))
# Set WD at ROOT
os.chdir(root_path)
# Print to verify
print(f"Current working directory: {os.getcwd()}")

Current working directory: c:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow


In [2]:
# Utilities
from utils.load_yaml_prompt import load_yaml_prompt
from utils.load_vector_query import load_vector_query
from utils.parse_llm_json import parse_llm_json

In [3]:
# Import Nodes
from nodes.gatekeeper_node import gatekeeper_node
from nodes.reasearch_question_node import research_question_node
from nodes.data_understanding_node import data_understanding_node
from nodes.data_preprocessing_node import data_preprocessing_node
from nodes.modelling_node import modelling_node
from nodes.evaluation_nodes.evaluation_metrics_foundational import evaluation_metrics_foundational_node
from nodes.evaluation_nodes.evaluation_metrics_specialised import evaluation_metrics_specialised_node
from nodes.evaluation_nodes.evaluation_theoretical_orientation import evaluation_theoretical_orientation_node
from nodes.evaluation_nodes.evaluation_interpretability import evaluation_interpretability_node
from nodes.evaluation_nodes.evaluation_ethical_social import evaluation_ethical_social_node

In [4]:
from utils.dsrp_state import DSRPState

In [35]:
# Define function
def route_after_gatekeeper(state: DSRPState):
    decision = str(state.get("gatekeeper", {}).get("final_classification", "")).strip().lower()
    if decision == "include":
        return "workflow_start"
    # Stop workflow for Exclude and Borderline (or any non-Include decision).
    return END

In [21]:
def route_after_specialised(state: DSRPState):
    if not state["modelling"]["specialised_paradigms"]:
        return END
    return "modelling_sub_specialised"

In [36]:
# parallel graph with gatekeeper-controlled entry
from langgraph.graph import StateGraph, END

builder = StateGraph(DSRPState)

builder.add_node("gatekeeper", gatekeeper_node)
builder.add_node("workflow_start", lambda state: {})
builder.add_node("workflow_complete", lambda state: {})
builder.add_node("research_question", research_question_node)
builder.add_node("data_understanding", data_understanding_node)
builder.add_node("data_preprocessing", data_preprocessing_node)
builder.add_node("modelling", modelling_node)
builder.add_node("evaluation_metrics_foundational_node", evaluation_metrics_foundational_node)
builder.add_node("evaluation_theoretical_orientation_node", evaluation_theoretical_orientation_node)
builder.add_node("evaluation_interpretability_node", evaluation_interpretability_node)
builder.add_node("evaluation_ethical_social_node", evaluation_ethical_social_node)

builder.set_entry_point("gatekeeper")

# Gatekeeper decides whether workflow continues.
builder.add_conditional_edges(
    "gatekeeper",
    route_after_gatekeeper
)

# Parallel fan-out starts immediately after Include.
builder.add_edge("workflow_start", "research_question")
builder.add_edge("workflow_start", "data_understanding")
builder.add_edge("workflow_start", "data_preprocessing")
builder.add_edge("workflow_start", "modelling")
builder.add_edge("workflow_start", "evaluation_theoretical_orientation_node")
builder.add_edge("workflow_start", "evaluation_interpretability_node")
builder.add_edge("workflow_start", "evaluation_ethical_social_node")

# Foundational metrics depends on modelling output.
builder.add_edge("modelling", "evaluation_metrics_foundational_node")

# Join all active branches before END.
builder.add_edge("research_question", "workflow_complete")
builder.add_edge("data_understanding", "workflow_complete")
builder.add_edge("data_preprocessing", "workflow_complete")
builder.add_edge("evaluation_metrics_foundational_node", "workflow_complete")
builder.add_edge("evaluation_theoretical_orientation_node", "workflow_complete")
builder.add_edge("evaluation_interpretability_node", "workflow_complete")
builder.add_edge("evaluation_ethical_social_node", "workflow_complete")

builder.add_edge("workflow_complete", END)

graph = builder.compile()

Test State

In [37]:
COLLECTION_NAME = "theory_synthesis"
PERSIST_DIRECTORY = "./vector_database"
embedding_model = "text-embedding-3-small"
PAPER_FOLDER = "./my_papers"

In [38]:
from utils.dsrp_state import DSRPState

test_state: DSRPState = {
    "paper_id": "2024 - 15.pdf",
    "gatekeeper": {},
    "dsrp_outputs": {},
    "collection_name": COLLECTION_NAME,
    "persist_directory": PERSIST_DIRECTORY,
    "embedding_model": embedding_model
}


In [39]:
result = graph.invoke(test_state)
print(result)

{'paper_id': '2024 - 15.pdf', 'dsrp_outputs': {'evaluation_ethical_social': {'privacy_protection_reported': 'No', 'bias_fairness_considered': 'No', 'societal_impact_discussed': 'No', 'ethical_reflection_level': 'Low', 'confidence': 1.0, 'validated_reasoning': 'The study does not provide any evidence of privacy protection measures, does not discuss bias or fairness issues, and does not address potential societal impacts. Therefore, it reflects a low level of ethical reflection.', 'validated_bibliography': [], 'audit_commentary': 'All classifications are supported by the lack of evidence in the study.'}, 'data_preprocessing': {'validated_data_cleaning': {'status': 'Transparently Described', 'evidence_ids': [1]}, 'validated_data_reduction': {'status': 'Not Reported', 'evidence_ids': []}, 'validated_data_transformation': {'status': 'Mentioned', 'evidence_ids': [1]}, 'confidence': 0.9, 'validated_reasoning': 'Data cleaning is transparently described with specific actions taken (removal of r

In [40]:
import json
from html import escape
from IPython.display import HTML, display

cards = []
for node, payload in result.get("dsrp_outputs", {}).items():
    confidence = payload.get("confidence", "-")
    summary = (
        payload.get("final_classification")
        or payload.get("evaluation_quality")
        or payload.get("foundational_paradigm")
        or ""
    )
    cards.append(
        f"""
        <details class='card'>
          <summary><b>{escape(node)}</b> | confidence: {escape(str(confidence))} {escape(str(summary))}</summary>
          <pre>{escape(json.dumps(payload, indent=2, ensure_ascii=False))}</pre>
        </details>
        """
    )

display(HTML(
    """
    <style>
      .grid {display:grid; gap:10px;}
      .card {border:1px solid #d7d7d7; border-radius:8px; padding:8px 10px; background:#fafafa;}
      summary {cursor:pointer;}
      pre {white-space:pre-wrap; margin:8px 0 0; font-size:12px;}
    </style>
    <h3>DSRP Results</h3>
    <div class='grid'>""" + "".join(cards) + "</div>"
))